In [1]:
import torch
from torch.nn import Module, ModuleList, Parameter, Buffer
import tiktoken
import math
import os
import re
import numpy as np
from collections import Counter
print('Hello World')

Hello World


# Data Loader

We will build a data loader that loads (pretokenized) chat conversations and returns them as data suitable for training. 


- Now let's implement a DataLoader class that will serve as the way to feed tokens. This setup will only read the tokens from a file as they are needed, rather than read the whole file into memory first (to be fair, in this current version the max size of the tokens in memory is just 1GB, so it would be possible to keep it all in memory, but these are still useful techniques).

We should implement this as a Python iterator which means that in addition to the __init__() function, we have to implement an __iter__() function (which resets the iteration), and a __next__() function that return the next element in the dataset.

The approach should work like this:

1. On initialization, we should save the filename (or, if you prefer, open a file object, it doesn't really matter here), any other elements we need to store within the class, and compute the number of batches total in the dataset -- this number will be the total size of the file divided by `seq_len * batch_size * 2` where this is due to the fact that each batch has `batch_size` token sequences each of size `seq_len`, and the two factor comes from the fact that the each token is an unsigned 16 bit number (two bytes).

    \begin{bmatrix}
    t_1 & t_2 & t_3 & t_4 \\
    t_5 & t_6 & t_7 & t_8
    \end{bmatrix}

    - for this batch, batch_size = 2, seq_len = 4

2. The __iter__() function should reset the batch counter and return `self`.
3. The __next__() function should open the file (if not open already, but be sure it close it so you don't open too many files), use `.seek()` to reach the point in the file for the current batch (***current_batch * seq_len * batch_size * 2***), then read ***(seq_len+1) * batch_size*** 16 bit tokens (one more than ***seq_len*** for each batch, because we want to load both the current and next token target for the whole sequence). You can read tokens by reading the proper number of types and then calling the command:

- `torch.tensor(np.frombuffer(bytes,dtype=np.uint16).astype(np.int64))`

- You should resize this tensor to be a `batch_size x seq_len+1`, and call `.to(<device>)` on the tensor, where `<device>` is whatever parameter is passed to the initializer as device. Finally, return the first seq_len tokens and the next seq_len tokens shifted by 1 as the `X,Y` return values of this loader. Then increment the current batch_counter (if this number of larger than the number of batches, i.e., there is not enough data left to read then raise `StopIteration`).

In [2]:
class DataLoader:
    def __init__(self,filename, seq_len, batch_size, device="cpu"):
        """
        Initialize a sequential token data loader backed by a binary file.
        Inputs:
            filename: str - binary filename containing unit16 token ids (i.e. out_filename from pretokenize_data)
            seq_len: int - number of tokens per input sequence (each row is a sequence, so how many columns in each row)
            batch_size: int - number of sequences per minibatch (how many row in each batch)
            device: str - device on which to place each minibatch tensor
        """
        self.filename = filename
        self.seq_len = seq_len
        self.batch_size = batch_size
        self.device = device
        bytes_per_batch = batch_size * seq_len * 2
        file_size = os.path.getsize(filename)
        self.num_batches_tot = file_size // bytes_per_batch

    def __iter__(self):
        """
        Reset iteration state and return the iterator object.
        Output:
            DataLoader - iterate over token minimatches
        """
        self.current_batch = 0 ### Each time you start a new epoch, you need to go back to the beginning.
        return self
    
    def __next__(self):
        """
        Return the next input-target token minibatch.
        Output:
            tuple(torch.Tensor, torch.Tensor) - current input tokens and next-token targets
        """
        if self.current_batch >= self.num_batches_tot:
            raise StopIteration
        with open(self.filename, 'rb') as f: ### 'rb' => read binary
            f.seek(self.current_batch * self.seq_len * self.batch_size * 2)
            num_bytes = (self.seq_len + 1) * self.batch_size * 2 ### bytes to read
            raw_data = f.read(num_bytes)
            data = torch.tensor(np.frombuffer(raw_data,dtype=np.uint16).astype(np.int64))

        data = data.reshape(self.batch_size, self.seq_len+1) ### 1D (batch_size * (seq_len + 1) => 2D (batch_size, seq_len+1))
        data = data.to(self.device)

        X = data[:, :self.seq_len]
        Y = data[:, 1:] ### next seq_len tokens shifted by 1

        self.current_batch += 1

        return X, Y